# Комбинация 7: Hierarchical Supervisor + Router

Корпоративный AI-ассистент. Верхний супервайзер распределяет задачи между отделами (HR, IT, Finance). IT-отдел — подграф с внутренним роутером, который одноразово направляет запрос к нужному специалисту.

Двухуровневая иерархия: верхний супервайзер определяет отдел, IT-роутер внутри определяет специалиста. Для HR и Finance роутер не нужен — запросы достаточно однородны. Для IT, где десятки категорий, внутренний роутер необходим. Это принцип **точечного усиления**: сложность только там, где она оправдана.

```mermaid
graph LR
    START((START)) --> supervisor{{"🎯 Supervisor<br/><i>распределяет</i>"}}
    supervisor -->|hr| hr(["🔹 HR-агент<br/><i>обрабатывает</i>"])
    supervisor -->|it| it_router{{"🎯 IT Router<br/><i>классифицирует</i>"}}
    supervisor -->|finance| finance(["🔹 Finance-агент<br/><i>обрабатывает</i>"])
    it_router -->|network| network(["🔹 Сетевой специалист<br/><i>обрабатывает</i>"])
    it_router -->|security| security(["🔹 Специалист по безопасности<br/><i>обрабатывает</i>"])
    it_router -->|infra| infra(["🔹 Инфраструктурный инженер<br/><i>обрабатывает</i>"])
    hr --> END((END))
    network --> END
    security --> END
    infra --> END
    finance --> END

    classDef coordinator fill:#4A90D9,stroke:#2C5F8A,color:#fff,stroke-width:2px
    classDef worker fill:#2ECC71,stroke:#1A8B4C,color:#fff,stroke-width:2px
    classDef aggregator fill:#F59E0B,stroke:#D97706,color:#fff,stroke-width:2px
    classDef terminal fill:#95A5A6,stroke:#707B7C,color:#fff

    class START,END terminal
    class supervisor,it_router coordinator
    class hr,finance,network,security,infra worker
```

In [1]:
import sys
sys.path.insert(0, "..")

import json
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
from llm_config import get_llm, check_api_key

In [2]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

API key is set


## Состояния двух уровней

На верхнем уровне `TopState` содержит запрос, определённый домен (hr / it / finance) и итоговый ответ. IT-подграф работает с собственным `ITState`, где вместо домена — категория специалиста (network / security / infra). LangGraph автоматически пробрасывает совпадающие по имени поля при вызове подграфа.

In [3]:
llm = get_llm()

class TopState(TypedDict):
    query: str       # Запрос сотрудника
    domain: str      # Целевой отдел (hr / it / finance)
    response: str    # Ответ от отдела

class ITState(TypedDict):
    query: str       # Запрос к IT
    category: str    # Выбранный специалист (network / security / infra)
    response: str    # Ответ специалиста

## Подграф IT-отдела с роутером

IT-отдел — это отдельный подграф с паттерном Router внутри. Роутер классифицирует запрос одним вызовом LLM и направляет к нужному специалисту: сетевому инженеру, специалисту по безопасности или инфраструктурному инженеру. Одноразовая маршрутизация без цикла — запрос сразу попадает к эксперту.

In [4]:
def it_router(state: ITState) -> dict:
    """Роутер IT-отдела: определяет специалиста."""
    response = llm.invoke([
        SystemMessage(content="""Классифицируй IT-запрос. Ответь ОДНИМ словом:
- network — проблемы с сетью, VPN, подключением
- security — безопасность, доступы, пароли
- infra — серверы, деплой, инфраструктура"""),
        HumanMessage(content=state["query"])
    ])
    cat = response.content.strip().lower()
    if cat not in ("network", "security", "infra"):
        cat = "infra"
    print(f"    [IT Router] → {cat}")
    return {"category": cat}


def network_spec(state: ITState) -> dict:
    """Сетевой специалист: VPN, DNS, маршрутизация."""
    response = llm.invoke([
        SystemMessage(content="Ты — сетевой специалист. Помоги с сетевой проблемой. 2-3 предложения."),
        HumanMessage(content=state["query"])
    ])
    return {"response": response.content}


def security_spec(state: ITState) -> dict:
    """Специалист по безопасности: доступы, сертификаты, инциденты."""
    response = llm.invoke([
        SystemMessage(content="Ты — специалист по безопасности. 2-3 предложения."),
        HumanMessage(content=state["query"])
    ])
    return {"response": response.content}


def infra_spec(state: ITState) -> dict:
    """Инфраструктурный инженер: серверы, деплой, мониторинг."""
    response = llm.invoke([
        SystemMessage(content="Ты — инфраструктурный инженер. 2-3 предложения."),
        HumanMessage(content=state["query"])
    ])
    return {"response": response.content}

## Компиляция IT-подграфа

Собираем подграф IT-отдела: `START → router → (network | security | infra) → END`. Условное ребро из роутера направляет к специалисту на основе поля `category`. Каждый специалист — терминальный узел, ведущий к `END`.

In [5]:
def it_route(state: ITState) -> str:
    return state["category"]

it_graph = StateGraph(ITState)
it_graph.add_node("router", it_router)
it_graph.add_node("network", network_spec)
it_graph.add_node("security", security_spec)
it_graph.add_node("infra", infra_spec)

it_graph.add_edge(START, "router")
it_graph.add_conditional_edges("router", it_route, {
    "network": "network",
    "security": "security",
    "infra": "infra",
})
it_graph.add_edge("network", END)
it_graph.add_edge("security", END)
it_graph.add_edge("infra", END)

it_compiled = it_graph.compile()
print("IT-подграф скомпилирован")

IT-подграф скомпилирован


## Верхний уровень: Hierarchical Supervisor

Главный супервайзер определяет домен запроса (hr / it / finance) и направляет к соответствующему отделу. HR и Finance — простые агенты с прямым ответом. IT-отдел — скомпилированный подграф с роутером внутри, подключённый как обычный узел через `it_compiled`.

In [6]:
def top_supervisor(state: TopState) -> dict:
    """Верхний супервайзер: определяет отдел."""
    response = llm.invoke([
        SystemMessage(content="Определи домен запроса. Ответь ОДНИМ словом: hr, it, finance."),
        HumanMessage(content=state["query"])
    ])
    domain = response.content.strip().lower()
    if domain not in ("hr", "it", "finance"):
        domain = "it"
    print(f"  [Гл. супервайзер] Домен: {domain}")
    return {"domain": domain}


def hr_agent(state: TopState) -> dict:
    """HR-специалист: кадровые вопросы, отпуска, зарплата."""
    response = llm.invoke([
        SystemMessage(content="Ты — HR-специалист. Ответь на вопрос. 2-3 предложения."),
        HumanMessage(content=state["query"])
    ])
    print(f"  [HR] Ответ готов")
    return {"response": response.content}


def finance_agent(state: TopState) -> dict:
    """Финансовый специалист: бюджеты, отчётность, закупки."""
    response = llm.invoke([
        SystemMessage(content="Ты — финансовый специалист. Ответь на вопрос. 2-3 предложения."),
        HumanMessage(content=state["query"])
    ])
    print(f"  [Finance] Ответ готов")
    return {"response": response.content}

## Сборка основного графа

Условное ребро из супервайзера направляет к нужному отделу. Обратите внимание: IT-узел — это `it_compiled`, скомпилированный подграф. LangGraph вызывает его как чёрный ящик, пробрасывая совпадающие поля состояния (`query`, `response`). Все три ветки — терминальные, ведут к `END`.

In [7]:
def top_route(state: TopState) -> str:
    return state["domain"]

top_graph = StateGraph(TopState)
top_graph.add_node("supervisor", top_supervisor)
top_graph.add_node("hr", hr_agent)
top_graph.add_node("it", it_compiled)       # подграф с Router внутри
top_graph.add_node("finance", finance_agent)

top_graph.add_edge(START, "supervisor")
top_graph.add_conditional_edges("supervisor", top_route, {
    "hr": "hr",
    "it": "it",
    "finance": "finance",
})
top_graph.add_edge("hr", END)
top_graph.add_edge("it", END)
top_graph.add_edge("finance", END)

app = top_graph.compile()
print("Основной граф скомпилирован")

Основной граф скомпилирован


## Запуск

Тестируем на трёх запросах из разных доменов: сетевая проблема (IT → network), кадровый вопрос (HR), инфраструктурная задача (IT → infra). В логах видно, как главный супервайзер направляет в отдел, а IT-роутер внутри выбирает специалиста.

In [8]:
queries = [
    "Не работает VPN из дома",
    "Когда следующая индексация зарплат?",
    "Нужно обновить сертификат на продакшн-сервере",
]

for q in queries:
    print(f"\n{'=' * 60}")
    print(f"Запрос: {q}")
    print("=" * 60)
    result = app.invoke({"query": q, "domain": "", "response": ""})
    print(f"Ответ: {result['response'][:200]}...")


Запрос: Не работает VPN из дома


  [Гл. супервайзер] Домен: it


    [IT Router] → network


Ответ: Проверьте сначала, не блокирует ли домашний роутер или провайдер сам VPN: попробуйте подключиться с мобильного интернета и сравните. Затем убедитесь, что на домашнем Wi‑Fi нет фильтрации UDP 500/4500 ...

Запрос: Когда следующая индексация зарплат?


  [Гл. супервайзер] Домен: hr


  [HR] Ответ готов
Ответ: Следующая индексация зарплат обычно проводится по итогам установленного в компании периода — чаще всего раз в год, но точная дата зависит от внутреннего графика и бюджета. Если хотите, я могу уточнить...

Запрос: Нужно обновить сертификат на продакшн-сервере


  [Гл. супервайзер] Домен: it


    [IT Router] → infra


Ответ: Сделаю, но нужны детали: какой именно сертификат и где он используется — nginx/apache, балансировщик, ingress или приложение напрямую?  
Также пришлите путь к текущему сертификату/ключу, новый сертифи...


## Итоги

**Hierarchical Supervisor + Router** --- двухуровневая архитектура для корпоративных систем:

- **Верхний уровень (Supervisor):** главный супервайзер определяет отдел и делегирует задачу. Простые отделы (HR, Finance) обслуживаются одним агентом.
- **Нижний уровень (Router):** IT-отдел содержит внутренний роутер, который одноразово классифицирует запрос и направляет к специалисту. Никакого цикла --- один вызов LLM для маршрутизации.
- **Принцип точечного усиления:** сложность добавляется только там, где она оправдана. IT-отдел с десятками категорий нуждается в роутере; HR и Finance --- нет.
- **Подграф как чёрный ящик:** `it_compiled` подключается к основному графу как обычный узел. LangGraph пробрасывает совпадающие поля состояния автоматически.